In [ ]:
EXP_NAME="synthetic_dataset"
N = 10000
n_samples = 3000
seed = 4
minority_perc = 0.3
prevalence = 5

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '..'

# Generate the clinical ground truth

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import skewnorm, gamma, beta
from scipy.special import expit

def generate_clinical_ground_truth(n_pop=10000, seed=42, sens_split=0.3):
    """
    Generates unbiased health states and legitimate correlations.
    """
    np.random.seed(seed)

    # ---- BIOLOGICAL ANCHORS ----
    
    # SENSITIVE ATTRIBUTE
    # 0 = Minority, 1 = Majority
    S = np.random.binomial(1, 1-sens_split, n_pop)

    # AGE
    age = np.random.normal(65, 12, n_pop).round(0)

    # GENETIC ANTECEDENTS: 
    # Count of risk alleles (0, 1, 2)
    # Minor Allele Frequency (MAF) differs by ancestry proxy (S)
    maf = np.where(S == 0, 0.38, 0.12)
    genetic_marker = np.random.binomial(2, maf, n_pop)
    
    # LATENT 'pathology' drives both clinical manifestations and the target outcome
    # Influenced by Age and Genetics + rare Gamma noise
    age_std = (age - 65) /12
    gen_effect = (genetic_marker - np.mean(genetic_marker))
    # Base pathology is Gamma-distributed (rare/skewed)
    path_raw = gamma.rvs(a=1.5, scale=1.0, size=n_pop)
    # Total pathology: combination of linear risks and rare base state
    pathology = 0.5 * age_std + 0.8 * gen_effect + path_raw
    pathology = pathology - np.mean(pathology)

    # ---- CLINICAL OUTCOME (Only influenced by Pathology) ---
    # Intercept tuned to pre-defined prevalence
    logits = -13.2 + prevalence * pathology 
    outcome_Y = np.random.binomial(1, expit(logits), n_pop)
    
    # ---- UNBIASED FEATURES INDEPENDENT FROM S, manifestations of the pathology ----

    # Symptom 1: Ordinal [0-4], skewed right
    s1_base_p = np.array([0.45, 0.30, 0.15, 0.07, 0.03])
    s1_weights = np.exp(np.outer(pathology, np.arange(5) * 0.4)) * s1_base_p
    s1_probs = s1_weights / s1_weights.sum(axis=1, keepdims=True)
    symptom_1 = np.array([np.random.choice([0, 1, 2, 3, 4], p=p) for p in s1_probs])
    
    # Symptom 2: Categorical [0, 1, 2], correlated with outcome via latent pathology
    # f0 (Healthy/Normal): High baseline, drops off quickly as pathology increases
    f0 = 1.2 - 1.8 * pathology 
    # f1 (Mild/Atypical): Moderate baseline, relatively stable across pathology
    f1 = 0.5 - 0.2 * pathology 
    # f2 (Severe/Specific): Low baseline, increases sharply with pathology
    f2 = -1.5 + 1.5 * pathology 
    s2_logits = np.column_stack([f0, f1, f2])
    s2_probs = np.exp(s2_logits) / np.sum(np.exp(s2_logits), axis=1, keepdims=True)
    symptom_2 = np.array([np.random.choice([0, 1, 2], p=p) for p in s2_probs])

    # Biomarker 1: Normal distribution
    biomarker_1 = np.random.normal(100 + 8 * pathology, 15, n_pop)
    
    # Biomarker 2: Skewed right continuous (e.g., Enzyme levels)
    alpha_b2 = np.maximum(0.5, 2.0 + 0.7 * pathology)
    biomarker_2 = gamma.rvs(a=alpha_b2, scale=10, size=n_pop)
    
    # Risk score for procedure: Continuous, skewed right, bounded [0, 1]
    a_param = 1.5 + np.exp(0.8 * pathology) 
    b_param = 5.0
    risk_score_proc = beta.rvs(a=a_param, b=b_param, size=n_pop)

    # ---- FEATURES CORRELATED WITH S (fair bias) ----

    # Symptom 3: Binary, prevalence depends on sensitive attribute S
    p3_logits = -1.5 + 1.2 * (1-S) + 0.8 * pathology
    symptom_3 = np.random.binomial(1, expit(p3_logits), n_pop)

    # Symptom 4: Categorical [0, 1, 2], group-specific proportions
    s4_f0 = np.where(S == 0, 1.0, 0.2) - 1.0 * pathology
    s4_f1 = np.where(S == 0, 0.2, 0.8) + 0.2 * pathology
    s4_f2 = np.where(S == 0, 0.1, 0.5) + 0.8 * pathology
    s4_logits = np.column_stack([s4_f0, s4_f1, s4_f2])
    s4_probs = np.exp(s4_logits) / np.sum(np.exp(s4_logits), axis=1, keepdims=True)
    symptom_4 = np.array([np.random.choice([0, 1, 2], p=p) for p in s4_probs])

    # Biomarker 3: Normal, group-specific means
    biomarker_3 = 40 + 15*S + 5*pathology + np.random.normal(0, 10, n_pop)
    
    return pd.DataFrame({
      'S': S, 'age': age, 'genetic_marker': genetic_marker,
      'pathology_latent': pathology, 'symptom_1': symptom_1,
      'symptom_2': symptom_2, 'biomarker_1': biomarker_1,
      'biomarker_2': biomarker_2, 'risk_score_proc': risk_score_proc,
      'symptom_3': symptom_3, 'symptom_4': symptom_4,
      'biomarker_3': biomarker_3, 'outcome_Y': outcome_Y
    })

# Initialize the unbiased dataset
df_unbiased = generate_clinical_ground_truth()

# Verification of AFib Prevalence
print(f"Baseline Outcome Prevalence: {df_unbiased['outcome_Y'].mean():.2%}")
print(df_unbiased.describe())

# Introducing bias

## Selection bias

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def simulate_selection_bias(df, n_target=3000, seed=42):
  """
  Sub-samples the population to introduce selection bias.
  Minority participants (S=0) are more likely to be selected if they are older.
  """
  np.random.seed(seed)
  
  # Standardize age to create stable selection weights
  age_std = (df['age'] - df['age'].mean()) / df['age'].std()
  
  # Define selection weights
  # For S=1: Selection probability increases exponentially with age
  # For S=0: Selection probability decreases slightly with age
  weights = np.ones(len(df))
  
  mask_0 = (df['S'] == 0)
  mask_1 = (df['S'] == 1)
  
  # Weighting function: w = exp(beta * age_std)
  # beta=1.0 creates a strong bias toward older individuals in the majority group
  weights[mask_1] = np.exp(0.3 * age_std[mask_1])
  weights[mask_0] = np.exp(-0.1 * age_std[mask_0])

  fig, ax = plt.subplots(figsize=(6, 3))
  sns.lineplot(x=df.loc[mask_0, 'age'], y=weights[mask_0], ax=ax, label="Group 0 weighting", color="red")
  sns.lineplot(x=df.loc[mask_1, 'age'], y=weights[mask_1], ax=ax, label="Group 1 weighting", color="blue")
  plt.show()
  
  # Normalize weights to sum to 1
  prob_selection = weights / weights.sum()
  
  # Sample the observed dataset
  observed_indices = np.random.choice(df.index, size=n_target, replace=False, p=prob_selection)
  
  return df.loc[observed_indices].reset_index(drop=True)

# Run the selection
df_sampled = simulate_selection_bias(df_unbiased, n_target=n_samples)

## Systemic bias

In [ ]:
def apply_systemic_biases(df, reporting_threshold=2, medication_effect=15.0, referral_bias=0.3, missingness_rate=0.4):
    """
    Introduces unfair pathways into the observed clinical features.
    """
    # Create observed copies to preserve 'true' columns for the audit
    S = df['S'].values

    # 1. Symptom 1 (Reporting Bias): High values in S=0 are recorded lower
    # If reported > threshold, it's 'capped' or reduced for the minority group
    df['symptom_1_obs'] = df['symptom_1']
    mask_s1 = (S == 0) & (df['symptom_1_obs'] > reporting_threshold)
    df.loc[mask_s1, 'symptom_1_obs'] = df.loc[mask_s1, 'symptom_1_obs'] - 1

    # 2. Biomarker 1 (Treatment Bias): S=1 has higher access to regulating meds
    # Medication typically 'regulates' (lowers) a high biomarker toward normal range
    # We apply this only to the Majority group to simulate differential access
    df['biomarker_1_obs'] = df['biomarker_1']
    df.loc[S == 1, 'biomarker_1_obs'] = df.loc[S == 1, 'biomarker_1_obs'] - medication_effect

    # 3. Procedure (Referral Bias): S=0 less often referred despite high risk
    # We define a 'referred' binary variable based on the risk_score_proc
    # Majority (S=1) systematically referred if score > 0.5; 
    # Minority (S=0) needs score > 0.5 but referral are systematically denied
    base_referral = df['risk_score_proc'] > 0.5
    bias_mask = (S == 0) & (np.random.random(len(df)) < referral_bias)
    df['procedure'] = base_referral.astype(int)
    df.loc[bias_mask, 'procedure'] = 0 # Systemic denial of referral

    # 4. Symptom 3 (Mixed Pathway Bias): S=0 under-reports/never reports
    # A proportion of 'True' symptom_3 cases in group 0 are zeroed out
    df['symptom_3_obs'] = df['symptom_3']
    s3_report_mask = (S == 0) & (df['symptom_3_obs'] == 1) & (np.random.random(len(df)) < missingness_rate)
    df.loc[s3_report_mask, 'symptom_3_obs'] = 0

    df_obs = df.drop(['symptom_1', 'biomarker_1', 'risk_score_proc', 'symptom_3', 'pathology_latent'], axis=1)

    return df_obs

df_observed = apply_systemic_biases(df_sampled)
# print(df_observed.head())

# Data processing

## Study population summary

In [ ]:
from tableone import TableOne

# Descriptive statistics
table1 = TableOne(df_observed,
                  groupby='S',
                  continuous=['age','symptom_1_obs','biomarker_1_obs','biomarker_2','biomarker_3'],
                  categorical=['outcome_Y','genetic_marker', 'symptom_2','symptom_3_obs', 'symptom_4'],
                  missing=False,
                  sort=True
                  )

print(table1)

## Distributions

### Sociologically influenced features

In [ ]:
# Age
plt.figure(figsize=(10, 4))
sns.histplot(df_observed, x='age', hue="S", bins=50, common_norm=False, multiple='dodge', kde=True, stat='probability')
plt.title('Probability distribution of Age')
plt.show()

In [ ]:
# Observed Biomarker 1
plt.figure(figsize=(10, 4))
sns.histplot(df_observed, x='biomarker_1_obs', hue="S", bins=50, common_norm=False, multiple='dodge', kde=True, stat='probability')
plt.title('Probability distribution of Observed Biomarker 1')
plt.show()

In [ ]:
# Observed Symptom 1
plt.figure(figsize=(8, 4))
sns.histplot(df_observed, x='symptom_1_obs', hue="S", common_norm=False, multiple='dodge', discrete=True, stat='probability')
plt.title('Probability distribution of Observed Symptom 1')
plt.show()

In [ ]:
# Procedure
plt.figure(figsize=(4, 4))
sns.histplot(x=df_observed['procedure'].astype(str), hue=df_observed["S"], common_norm=False, multiple='dodge', discrete=True, stat='probability')
plt.title('Probability distribution of Procedure')
plt.show()

### S-correlated features

In [ ]:
# Genetic marker
plt.figure(figsize=(8, 4))
sns.histplot(df_observed, x='genetic_marker', hue="S", common_norm=False, multiple='dodge', discrete=True, stat='probability')
plt.title('Probability distribution of Genetic Marker')
plt.show()


In [ ]:
# Symptom 4
plt.figure(figsize=(8, 4))
sns.histplot(df_observed, x='symptom_4', hue="S", common_norm=False, multiple='dodge', discrete=True, stat='probability')
plt.title('Probability distribution of Symptom 4')
plt.show()

In [ ]:
# Biomarker 3
plt.figure(figsize=(10, 4))
sns.histplot(df_observed, x='biomarker_3', hue="S", bins=50, common_norm=False, multiple='dodge', kde=True, stat='probability')
plt.title('Probability distribution of Biomarker 3')
plt.show()

### Independent features

In [ ]:
# Symptom 2
plt.figure(figsize=(8, 4))
sns.histplot(df_observed, x='symptom_2', hue="S", common_norm=False, multiple='dodge', discrete=True, stat='probability')
plt.title('Probability distribution of Symptom 2')
plt.show()

In [ ]:
# Biomarker 2
plt.figure(figsize=(10, 4))
sns.histplot(df_observed, x='biomarker_2', hue="S", bins=50, common_norm=False, multiple='dodge', kde=True, stat='probability')
plt.title('Probability distribution of Biomarker 2')
plt.show()

## Pre-processing

In [ ]:
df_processed = df_observed.copy()

# Z-score for Age, Obs Biomarker 1, Biomarker 3
norm_variables = ['age', "biomarker_1_obs", "biomarker_3"]
for var in norm_variables:
  df_processed[var] = (df_processed[var] - df_processed[var].mean()) / df_processed[var].std()

# Log and Z-score for Biomarker 2
skewed_variables = ["biomarker_2"]
for var in skewed_variables:
  log_var = np.log(df_processed[var])
  df_processed[var] = (log_var - log_var.mean()) / log_var.std()

df_processed.reset_index(drop=True, inplace=True)

df_processed.to_csv(f'{PROJECT_ROOT}/data/{EXP_NAME}_cleaned.csv', index=False)